# The Logit Lens: What Does Your Model Think at Each Layer?

The **logit lens** (nostalgebraist, 2020) is one of the simplest and most powerful interpretability tools. The idea:

> At every layer, the residual stream lives in the same space as the final output. So we can project it through the unembedding matrix to see what the model "believes" at that layer.

This notebook walks through:
1. What the logit lens is and why it works
2. Applying it to GPT-2 Small
3. Visualizing convergence of predictions across layers
4. KL divergence from the final distribution
5. The **tuned lens** (Belrose et al., 2023) — a learned improvement

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk.logit_lens import (
    logit_lens, logit_lens_top_k, logit_lens_correct_prob,
    logit_lens_kl_divergence, train_tuned_lens,
)

print("Loading GPT-2 Small...")
model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Loaded: {model.cfg.n_layers} layers, d_model={model.cfg.d_model}, vocab={model.cfg.d_vocab}")

## 1. The Intuition

A transformer builds up its answer incrementally through the residual stream. At each layer, information is added to the residual:

```
residual_0 = embed(tokens) + pos_embed
residual_1 = residual_0 + attn_0(residual_0) + mlp_0(residual_0)
residual_2 = residual_1 + attn_1(residual_1) + mlp_1(residual_1)
...
logits = unembed(layer_norm(residual_final))
```

The logit lens simply applies that final step (`layer_norm` + `unembed`) to every intermediate residual.

In [ ]:
# Let's pick a sentence where GPT-2 should have clear predictions
text = "The Eiffel Tower is located in the city of"
tokens = model.to_tokens(text)
token_labels = [tokenizer.decode([t]) for t in np.array(tokens)]

print(f"Input: {text!r}")
print(f"Tokens ({len(tokens)}): {token_labels}")

# What does the model actually predict?
logits = model(tokens)
pred_token = int(jnp.argmax(logits[-1]))
print(f"\nModel prediction for next token: {tokenizer.decode([pred_token])!r}")

## 2. Applying the Logit Lens

Let's see what the model predicts at each layer. We'll look at the top prediction for the final position (next-token prediction).

In [ ]:
# Get top-k predictions at every layer and position
top_k_results = logit_lens_top_k(model, tokens, k=5)

# Show top prediction at the last position for each layer
print("Layer-by-layer predictions for the final position:")
print("=" * 60)

layer_names = ["embed"] + [f"layer {i}" for i in range(model.cfg.n_layers)]
for i, name in enumerate(layer_names):
    preds = top_k_results[i][-1]  # last position
    top_token = tokenizer.decode([preds[0][0]])
    top_prob = preds[0][1]
    top5 = [f"{tokenizer.decode([p[0]])!r}({p[1]:.2f})" for p in preds[:3]]
    print(f"{name:>10s}: {top_token!r:>10s} ({top_prob:.3f})  |  top-3: {', '.join(top5)}")

Notice how the prediction converges! Early layers might predict generic tokens, but by the later layers, the model has "figured out" the answer.

## 3. Visualizing Correct-Token Probability

For each position, let's track the probability assigned to the *correct* next token as it builds up across layers.

In [ ]:
correct_probs = logit_lens_correct_prob(model, tokens)
# correct_probs shape: [n_components, seq_len - 1]

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(correct_probs, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)

# Labels
y_labels = ["embed"] + [f"L{i}" for i in range(model.cfg.n_layers)]
x_labels = [f"{token_labels[j]}\n->{token_labels[j+1]}" for j in range(len(tokens)-1)]

ax.set_yticks(range(len(y_labels)))
ax.set_yticklabels(y_labels)
ax.set_xticks(range(len(x_labels)))
ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=8)
ax.set_xlabel("Position (current -> correct next token)")
ax.set_ylabel("Layer")
ax.set_title("Logit Lens: P(correct next token) by Layer and Position")
plt.colorbar(im, ax=ax, label="Probability")
plt.tight_layout()
plt.show()

## 4. KL Divergence from Final Layer

Another way to measure convergence: how different is each layer's prediction from the model's final output? We measure this with **KL divergence**.

- High KL = the layer's prediction is very different from the final answer
- Low KL = the layer has essentially converged to the final prediction

In [ ]:
kl_divs = logit_lens_kl_divergence(model, tokens)
# kl_divs shape: [n_components - 1, seq_len]

# Average KL across positions
mean_kl = np.mean(kl_divs, axis=1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: heatmap
im = ax1.imshow(kl_divs, aspect='auto', cmap='viridis')
y_labels = ["embed"] + [f"L{i}" for i in range(model.cfg.n_layers - 1)]
ax1.set_yticks(range(len(y_labels)))
ax1.set_yticklabels(y_labels)
ax1.set_xticks(range(len(token_labels)))
ax1.set_xticklabels(token_labels, rotation=45, ha='right', fontsize=8)
ax1.set_title("KL(final || layer) by Position")
ax1.set_ylabel("Layer")
plt.colorbar(im, ax=ax1, label="KL Divergence")

# Right: average convergence curve
ax2.plot(mean_kl, marker='o')
ax2.set_xticks(range(len(y_labels)))
ax2.set_xticklabels(y_labels, rotation=45)
ax2.set_xlabel("Layer")
ax2.set_ylabel("Mean KL Divergence")
ax2.set_title("Convergence to Final Prediction")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Comparing Multiple Prompts

Let's see how the logit lens behaves on different types of knowledge retrieval.

In [ ]:
prompts = [
    "The capital of France is",
    "Barack Obama was the 44th President of the United",
    "1 + 1 =",
    "To be or not to be, that is the",
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (prompt, ax) in enumerate(zip(prompts, axes.flat)):
    toks = model.to_tokens(prompt)
    tok_labels = [tokenizer.decode([t]) for t in np.array(toks)]
    
    correct_p = logit_lens_correct_prob(model, toks)
    
    # Just show probability of the model's own predicted next token at final position
    # across layers
    final_logits = model(toks)
    pred = int(jnp.argmax(final_logits[-1]))
    pred_str = tokenizer.decode([pred])
    
    # Get logit lens probs for the predicted token at last position
    probs = logit_lens(model, toks, return_probs=True)
    pred_probs_by_layer = probs[:, -1, pred]
    
    layer_labels = ["emb"] + [f"L{i}" for i in range(model.cfg.n_layers)]
    ax.plot(pred_probs_by_layer, marker='o', markersize=3)
    ax.set_title(f'"{prompt}" -> {pred_str!r}', fontsize=10)
    ax.set_ylabel("P(predicted token)")
    ax.set_ylim(-0.05, 1.05)
    ax.set_xticks(range(0, len(layer_labels), 2))
    ax.set_xticklabels(layer_labels[::2], fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("Logit Lens: When Does the Model Decide?", fontsize=14)
plt.tight_layout()
plt.show()

## 6. The Tuned Lens

The **tuned lens** (Belrose et al., 2023) observes that the raw logit lens is biased: it treats all layers equally, but the model's representations change in systematic ways across layers.

The fix: learn an **affine probe** per layer that "translates" that layer's representation into the space the unembedding expects. Each probe starts as the identity (so it starts as the logit lens) and is trained to minimize KL divergence with the final output.

Let's train a tuned lens on some data and compare it to the standard logit lens.

In [ ]:
# Generate some training data - various prompts
train_prompts = [
    "The quick brown fox jumps over the lazy",
    "In the beginning, there was nothing but darkness and",
    "The president of the United States lives in the",
    "Machine learning models are trained using gradient",
    "The largest planet in our solar system is",
    "Water boils at 100 degrees",
    "Shakespeare wrote many famous plays including Romeo and",
    "The speed of light is approximately 300000 kilometers per",
]

train_tokens = jnp.stack([model.to_tokens(p)[:12] for p in train_prompts])
print(f"Training tuned lens on {train_tokens.shape[0]} sequences of length {train_tokens.shape[1]}...")

result = train_tuned_lens(model, train_tokens, epochs=30, lr=1e-3, verbose=True)

In [ ]:
# Compare logit lens vs tuned lens on a test prompt
test_prompt = "The Eiffel Tower is located in the city of"
test_tokens = model.to_tokens(test_prompt)
test_labels = [tokenizer.decode([t]) for t in np.array(test_tokens)]

# Standard logit lens
ll_probs = logit_lens(model, test_tokens, return_probs=True)

# Tuned lens
tl_probs = result.tuned_lens.apply(test_tokens, return_probs=True)

# Compare: probability of the correct next token at the final position
final_logits = model(test_tokens)
pred = int(jnp.argmax(final_logits[-1]))
pred_str = tokenizer.decode([pred])

ll_pred_probs = ll_probs[:, -1, pred]
tl_pred_probs = tl_probs[:, -1, pred]

fig, ax = plt.subplots(figsize=(10, 5))
layer_labels = ["emb"] + [f"L{i}" for i in range(model.cfg.n_layers)]
x = range(len(layer_labels))

ax.plot(x, ll_pred_probs, 'o-', label='Logit Lens', alpha=0.7)
ax.plot(x, tl_pred_probs, 's-', label='Tuned Lens', alpha=0.7)
ax.set_xticks(range(0, len(layer_labels), 2))
ax.set_xticklabels(layer_labels[::2])
ax.set_xlabel("Layer")
ax.set_ylabel(f"P({pred_str!r})")
ax.set_title(f"Logit Lens vs Tuned Lens: P(predicted token) at Each Layer")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

## Key Takeaways

1. **The residual stream is a shared communication channel** — all layers read from and write to the same space, which is why the logit lens works at all.

2. **Predictions converge over layers** — early layers are noisy, but by the middle layers, the model often "knows" its answer.

3. **Different types of knowledge appear at different layers** — simple syntactic predictions may converge early, while factual knowledge often requires later layers.

4. **KL divergence measures convergence** — it quantifies exactly how much each layer's belief differs from the final output.

5. **The tuned lens is more faithful** — by learning a per-layer affine transformation, it can give more accurate readings of what each layer "thinks."

### API Reference

```python
from irtk.logit_lens import (
    logit_lens,              # [n_layers+1, seq, vocab] logits at each layer
    logit_lens_top_k,        # Top-k token predictions per layer/position
    logit_lens_correct_prob, # P(correct next token) at each layer/position
    logit_lens_kl_divergence,# KL(final || layer) divergence
    train_tuned_lens,        # Train per-layer affine probes
)
```